# Health Eat — Experiment Ready Train → Predict → Submit

Q3/Q4 실험용으로 config 생성, 모델명 반영 패치, best checkpoint 선택, submission 백업까지 한 번에 진행하는 노트북입니다.


## 1. 환경 설정

In [ ]:
import os

REPO_URL   = "https://github.com/beomjinkim2000/Code_IT_Team_1_FirstProject.git"
PROJECT_DIR = "/content/Code_IT_Team_1_FirstProject"
BRANCH     = "main"  # 실험 브랜치 변경 시 여기만 수정

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} checkout {BRANCH}
    !git -C {PROJECT_DIR} pull origin {BRANCH}

%cd {PROJECT_DIR}

In [ ]:
!pip install -q \
    ultralytics \
    torchmetrics[detection] \
    albumentations \
    kaggle \
    pyyaml \
    tqdm \
    iterative-stratification \
    ensemble-boxes

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Kaggle 인증

**최초 1회 설정** (이후 재실행 시 자동 적용)

1. 왼쪽 사이드바 **🔑 아이콘** 클릭
2. **`KAGGLE_USERNAME`** → Kaggle 닉네임 입력 후 저장
3. **`KAGGLE_KEY`** → [Kaggle 계정](https://www.kaggle.com/settings/account) → API → **Create New Token** → kaggle.json 열어서 `key` 값 복사 후 저장
4. 각 Secret의 **노트북 액세스 허용** 토글 ON

In [ ]:
import json
import os
from google.colab import userdata

kaggle_username = userdata.get("KAGGLE_USERNAME")
kaggle_key      = userdata.get("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = kaggle_username
os.environ["KAGGLE_KEY"]      = kaggle_key

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
with open(f"{kaggle_dir}/kaggle.json", "w") as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod(f"{kaggle_dir}/kaggle.json", 0o600)

print(f"Kaggle 인증 완료 — 사용자: {kaggle_username}")

## 3. 데이터 다운로드

In [ ]:
from pathlib import Path
import zipfile

COMPETITION_ID = "ai11-level1-project"

raw_data_path = Path("data/raw/ai11-level1-project")
dataset_path  = raw_data_path / "sprint_ai_project1_data"
required_dirs = [
    dataset_path / "train_images",
    dataset_path / "test_images",
    dataset_path / "train_annotations",
]

if all(d.exists() for d in required_dirs):
    print("데이터 이미 존재 — 다운로드 건너뜀")
else:
    raw_data_path.mkdir(parents=True, exist_ok=True)
    !kaggle competitions download -c {COMPETITION_ID} -p {raw_data_path}
    for zf in raw_data_path.glob("*.zip"):
        with zipfile.ZipFile(zf) as z:
            z.extractall(raw_data_path)
        zf.unlink()
    print("압축 해제 완료")

print("\n데이터 구조:")
for d in required_dirs:
    count = len(list(d.glob("*"))) if d.exists() else 0
    print(f"  {d.name}: {count}개")

## 4. 학습

`configs/default.yaml`에서 설정 조정:
- `train.epochs`: 에폭 수 (기본 50)
- `train.batch_size`: 배치 크기 (기본 16, GPU 메모리 부족 시 8로 낮추기)
- `train.lr`: 학습률 (기본 0.001)

In [ ]:
!cat configs/default.yaml

## 4-1. Drive 마운트 및 실험 설정


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

SHARE_ROOT = Path('/content/drive/MyDrive/HealthEat/shared')
SHARE_ROOT.mkdir(parents=True, exist_ok=True)
print('share root:', SHARE_ROOT)


In [ ]:
# ── 실험 설정: Q3-B / Q4에서 주로 여기만 바꾸면 됩니다 ─────────────
EXP_NAME = "q3_yolo11x_e150_b8_lr0001_wd0001"
SUBMIT_VERSION = EXP_NAME
SUBMIT_MESSAGE = "Q3-B YOLO11x ep150 b8 lr1e-4 wd1e-4"

MODEL_NAME = "yolo11x"      # 예: yolov8x, yolo11x
EPOCHS = 150                 # Q4: 150 / 200 / 250
BATCH_SIZE = 8
IMG_SIZE = 640

PHASE1_LR = 0.0001
PHASE2_LR = 0.0001
PHASE3_HEAD_LR = 0.00001
PHASE3_BACKBONE_LR = 0.000001
PHASE3_LR_MIN = 0.000001
WEIGHT_DECAY = 0.0001

SKIP_TRAIN = False           # 이미 학습된 outputs/checkpoints가 있으면 True
USE_EMA_FOR_PREDICT = False  # 제출 생성 시 EMA 가중치 사용 여부

CONFIG_PATH = f"configs/{EXP_NAME}.yaml"
SHARE_DIR = SHARE_ROOT / EXP_NAME
SHARE_DIR.mkdir(parents=True, exist_ok=True)

print('EXP_NAME:', EXP_NAME)
print('CONFIG_PATH:', CONFIG_PATH)
print('SHARE_DIR:', SHARE_DIR)


In [ ]:
# config 생성 + model.name이 train/predict에 실제 반영되도록 소스 패치
from pathlib import Path
import yaml

repo = Path('/content/Code_IT_Team_1_FirstProject')
base_config = repo / 'configs/default.yaml'
config_path = repo / CONFIG_PATH

cfg = yaml.safe_load(base_config.read_text(encoding='utf-8-sig'))
cfg['model']['name'] = MODEL_NAME
cfg['train']['epochs'] = EPOCHS
cfg['train']['batch_size'] = BATCH_SIZE
cfg['train']['img_size'] = IMG_SIZE
cfg['train']['phase1_lr'] = PHASE1_LR
cfg['train']['phase2_lr'] = PHASE2_LR
cfg['train']['phase3_head_lr'] = PHASE3_HEAD_LR
cfg['train']['phase3_backbone_lr'] = PHASE3_BACKBONE_LR
cfg['train']['phase3_lr_min'] = PHASE3_LR_MIN
cfg.setdefault('optimizer', {})['weight_decay'] = WEIGHT_DECAY

config_path.write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True), encoding='utf-8')
print('config created:', config_path)

# build_model이 default.yaml이 아니라 현재 config의 model.name을 받게 패치
baseline_path = repo / 'src/models/baseline.py'
text = baseline_path.read_text(encoding='utf-8-sig')
text = text.replace(
    'def build_model(num_classes: int) -> torch.nn.Module:',
    'def build_model(num_classes: int, model_name: str | None = None) -> torch.nn.Module:',
)
text = text.replace(
    '    model_name = cfg["model"]["name"]',
    '    model_name = model_name or cfg["model"]["name"]',
)
baseline_path.write_text(text, encoding='utf-8')

for rel in ['train.py', 'predict.py']:
    path = repo / rel
    text = path.read_text(encoding='utf-8-sig')
    text = text.replace(
        'model = build_model(cfg["data"]["nc"])',
        'model = build_model(cfg["data"]["nc"], model_name=cfg["model"]["name"])',
    )
    path.write_text(text, encoding='utf-8')

print('patched build_model usage')
!grep -n "name:\|epochs:\|batch_size:\|phase1_lr:\|phase2_lr:\|phase3_head_lr:\|phase3_backbone_lr:\|weight_decay:" {CONFIG_PATH}
!python -m py_compile src/models/baseline.py train.py predict.py


In [ ]:
# 학습 실행
if SKIP_TRAIN:
    print('SKIP_TRAIN=True — 학습을 건너뜁니다')
else:
    !python -u train.py --config {CONFIG_PATH} --run-name {EXP_NAME}


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidate_paths = [
    Path(f'outputs/logs/metrics_{EXP_NAME}.csv'),
    Path(f'outputs/logs/metrics_{SUBMIT_VERSION}.csv'),
    Path('outputs/logs/metrics_default.csv'),
    Path('outputs/logs/metrics.csv'),
]
log_path = next((p for p in candidate_paths if p.exists()), None)

if log_path is None:
    print('metrics 파일 없음. outputs/logs 내용:')
    print(list(Path('outputs/logs').glob('*')) if Path('outputs/logs').exists() else 'outputs/logs 없음')
else:
    print('metrics:', log_path)
    df = pd.read_csv(log_path)
    display(df.tail(10))

    map_col = 'val_mAP_ema' if 'val_mAP_ema' in df.columns else ('val_mAP_raw' if 'val_mAP_raw' in df.columns else 'val_mAP')
    best = df.loc[df[map_col].idxmax()]
    print(f"best by {map_col}: epoch={int(best['epoch'])}, mAP={best[map_col]:.6f}")
    for col in ['val_mAP_50_ema', 'val_mAP_50_raw', 'val_box_loss', 'val_cls_loss', 'val_dfl_loss']:
        if col in df.columns:
            print(f'{col}: {best[col]:.6f}')

    plot_cols = [c for c in ['train_loss', 'box_loss', 'cls_loss', 'dfl_loss', 'val_mAP_raw', 'val_mAP_ema', 'val_mAP_50_raw', 'val_mAP_50_ema', 'lr'] if c in df.columns]
    if plot_cols:
        ax = df.set_index('epoch')[plot_cols].plot(subplots=True, figsize=(14, 3 * len(plot_cols)), grid=True)
        plt.tight_layout(); plt.show()


## 4-3. 클래스별 F1 추적

best epoch 기준 클래스별 F1을 확인합니다.

| 판단 기준 | 의미 |
|---|---|
| F1 < 0.7 (빨간색) | 해당 클래스가 bottleneck |
| Precision < Recall | 다른 클래스로 오탐(FP)이 많음 |
| Recall < Precision | 알약을 못 잡음(FN) — 데이터 부족 or 해상도 문제 |

→ Q4(YOLO11x), Q6(해상도 1024), Q7(AI Hub 데이터) 실험에서 어떤 클래스가 개선됐는지 비교하세요.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

f1_path = Path(f'outputs/logs/f1_{EXP_NAME}.csv')
metrics_path = Path(f'outputs/logs/metrics_{EXP_NAME}.csv')

if not f1_path.exists():
    print(f'f1 파일 없음: {f1_path}')
else:
    df_f1 = pd.read_csv(f1_path)
    if metrics_path.exists():
        df_m = pd.read_csv(metrics_path)
        metric_col = 'val_mAP_ema' if 'val_mAP_ema' in df_m.columns else 'val_mAP_raw'
        best_epoch = int(df_m.loc[df_m[metric_col].idxmax(), 'epoch'])
    else:
        best_epoch = int(df_f1['epoch'].max())

    df_best = df_f1[df_f1['epoch'] == best_epoch].copy()
    print('best_epoch:', best_epoch)
    display(df_best.sort_values('f1').head(20))

    if not df_best.empty:
        colors = ['red' if f < 0.7 else 'steelblue' for f in df_best['f1']]
        plt.figure(figsize=(10, max(4, len(df_best) * 0.25)))
        plt.barh(df_best['class_id'].astype(str), df_best['f1'], color=colors)
        plt.axvline(0.7, color='red', linestyle='--')
        plt.title(f'Class F1 at epoch {best_epoch}')
        plt.xlabel('F1'); plt.ylabel('class_id')
        plt.grid(True, alpha=0.3, axis='x')
        plt.show()


In [ ]:
import shutil
import subprocess
from pathlib import Path
import torch

ckpt_dir = Path('outputs/checkpoints')
sub_dir = Path('outputs/submissions')
drive_sub = Path('/content/drive/MyDrive/HealthEat/submissions')
sub_dir.mkdir(parents=True, exist_ok=True)
drive_sub.mkdir(parents=True, exist_ok=True)

def find_best_checkpoint():
    candidates = []
    for p in ckpt_dir.glob('epoch_*.pt'):
        ckpt = torch.load(p, map_location='cpu', weights_only=False)
        val = ckpt.get('val_mAP')
        if val is not None:
            candidates.append((float(val), p, ckpt.get('epoch')))
    if candidates:
        candidates.sort(reverse=True, key=lambda x: x[0])
        return candidates[0]
    for name in ['best_model.pt', 'best_model_ema.pt']:
        p = ckpt_dir / name
        if p.exists():
            return (None, p, None)
    raise FileNotFoundError('사용 가능한 checkpoint를 찾지 못했습니다')

best_map, BEST_CKPT, best_epoch = find_best_checkpoint()
print('BEST_CKPT:', BEST_CKPT)
print('best_epoch:', best_epoch, 'best_val_mAP:', best_map)

cmd = ['python', '-u', 'predict.py', '--config', CONFIG_PATH, '--checkpoint', str(BEST_CKPT)]
if USE_EMA_FOR_PREDICT:
    cmd.append('--use-ema')
print(' '.join(cmd))
subprocess.run(cmd, check=True)

src = sub_dir / 'submission_v1.csv'
model_label = 'ema' if USE_EMA_FOR_PREDICT else 'raw'
if src.exists():
    dst = sub_dir / f'submission_{EXP_NAME}_{model_label}.csv'
    shutil.copy2(src, dst)
    shutil.copy2(dst, drive_sub / dst.name)
    print('submission saved:', dst)
    print('drive copy:', drive_sub / dst.name)
else:
    print('submission_v1.csv 생성 실패')


In [ ]:
import pandas as pd
from pathlib import Path

sub_dir = Path('outputs/submissions')
for label, fname in [('EMA', f'submission_{SUBMIT_VERSION}_ema.csv'),
                     ('raw', f'submission_{SUBMIT_VERSION}_raw.csv')]:
    p = sub_dir / fname
    if p.exists():
        df = pd.read_csv(p)
        det = df.groupby('image_id').size()
        print(f"[{label:>3}]  행 수: {len(df):5d}  이미지: {df['image_id'].nunique():4d}"
              f"  탐지/이미지: {det.mean():.2f}  avg score: {df['score'].mean():.4f}"
              f"  max: {df['score'].max():.4f}  min: {df['score'].min():.4f}")
    else:
        print(f'[{label:>3}]  파일 없음: {p}')


## 5-1. 예측 진단

score 분포와 이미지당 탐지 수를 확인합니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

model_label = 'ema' if USE_EMA_FOR_PREDICT else 'raw'
csv_path = Path(f'outputs/submissions/submission_{EXP_NAME}_{model_label}.csv')
if not csv_path.exists():
    csv_path = Path('outputs/submissions/submission_v1.csv')

if not csv_path.exists():
    print('submission 파일 없음 — 예측 셀을 먼저 실행하세요')
else:
    df = pd.read_csv(csv_path)
    display(df.head())
    print('file:', csv_path)
    print('rows:', len(df), 'images:', df['image_id'].nunique())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df['score'], bins=50, edgecolor='black', color='steelblue')
    axes[0].axvline(0.25, color='red', linestyle='--', label='conf=0.25')
    axes[0].set_title('Score Distribution'); axes[0].legend()

    det_per_img = df.groupby('image_id').size()
    axes[1].hist(det_per_img, bins=range(0, int(det_per_img.max()) + 2), edgecolor='black')
    axes[1].set_title('Detections per Image')
    plt.tight_layout(); plt.show()

    print(f"score mean={df['score'].mean():.4f}, min={df['score'].min():.4f}, max={df['score'].max():.4f}")
    print(f"detections/image mean={det_per_img.mean():.2f}, max={det_per_img.max()}")


In [ ]:
import sys; sys.path.insert(0, '.')
from pathlib import Path
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches

from src.utils.config import load_config
from src.data.dataset import PillDataset, RAW_DATA_ROOT
from src.data.split import build_split_metadata, train_val_split
from src.data.transforms import val_transform
from src.engine.predict import predict_batch
from src.engine.postprocess import PostprocessConfig, postprocess_raw_outputs
from src.models.baseline import build_model

cfg = load_config(CONFIG_PATH)
img_size = cfg['train']['img_size']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt = torch.load(BEST_CKPT, map_location=device, weights_only=False)
state_key = 'ema_state' if USE_EMA_FOR_PREDICT and ckpt.get('ema_state') is not None else 'model_state'
model = build_model(cfg['data']['nc'], model_name=cfg['model']['name'])
model.load_state_dict(ckpt[state_key])
model.to(device).eval()

annotations = PillDataset.load_annotations()
category_to_label = cfg['data']['category_to_label']
labels_by_id, image_id_by_file = build_split_metadata(annotations, category_to_label)
all_files = sorted((RAW_DATA_ROOT / 'train_images').glob('*.png'))
_, val_files = train_val_split(
    all_files,
    val_ratio=cfg['split']['val_ratio'],
    seed=cfg['train']['seed'],
    method=cfg['split'].get('method', 'random'),
    labels_by_id=labels_by_id,
    image_id_by_file=image_id_by_file,
    num_classes=cfg['data']['nc'],
)
val_names = [p.name for p in val_files]
val_ds = PillDataset(
    split='val',
    transforms=val_transform(img_size),
    annotations=annotations,
    image_files=val_names[:4],
    category_to_label=category_to_label,
)

samples = [val_ds[i] for i in range(min(4, len(val_ds)))]
images = [s[0] for s in samples]
targets = [s[1] for s in samples]

with torch.no_grad():
    batch = torch.stack(images).to(device)
    raw = predict_batch(model, batch, device)
    preds = postprocess_raw_outputs(
        raw,
        image_ids=[t['image_id'] for t in targets],
        config=PostprocessConfig(**cfg['postprocess']),
    )

fig, axes = plt.subplots(1, len(samples), figsize=(5 * len(samples), 5))
if len(samples) == 1:
    axes = [axes]
for ax, img_tensor, target, pred in zip(axes, images, targets, preds):
    ax.imshow(img_tensor.permute(1, 2, 0).cpu())
    for box, label in zip(target['boxes'].cpu(), target['labels'].cpu()):
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='lime', facecolor='none'))
        ax.text(x1, max(0, y1-5), f'GT {int(label)}', color='lime', fontsize=7, backgroundcolor='black')
    for box, score, label in zip(pred['boxes'].cpu(), pred['scores'].cpu(), pred['labels'].cpu()):
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(x1, y1+8, f'P {int(label)} {float(score):.2f}', color='red', fontsize=7, backgroundcolor='white')
    ax.set_title(f"image_id={target['image_id']}")
    ax.axis('off')
plt.suptitle('Validation samples: green=GT, red=Prediction')
plt.tight_layout(); plt.show()


## 5-2. F1 곡선 — 최적 confidence threshold 찾기

현재 `postprocess.conf_threshold=0.25`는 관행적 기본값입니다.
이 셀은 val set 전체를 conf=0.001로 예측한 뒤 conf를 0.05→0.90으로 sweep해서
**F1이 최대인 conf**를 찾습니다.

- `best_conf`가 현재값과 ≥ 0.05 차이나면 → `configs/default.yaml`에 반영 후 재제출
- `skip_box_thr` (WBF)도 같은 방식으로 최적화 가능 ([[concepts/Confidence Score]])

In [ ]:
from src.engine.evaluate import compute_per_class_f1
from src.engine.postprocess import PostprocessConfig, postprocess_raw_outputs
from src.utils.collate import collate_fn
from torch.utils.data import DataLoader
import numpy as np
import torch
import matplotlib.pyplot as plt

nc = cfg['data']['nc']
current_conf = cfg['postprocess']['conf_threshold']
raw_post_cfg = PostprocessConfig(
    conf_threshold=0.001,
    iou_threshold=cfg['postprocess']['iou_threshold'],
    max_detections=cfg['postprocess']['max_detections'],
    class_agnostic_nms=cfg['postprocess'].get('class_agnostic_nms', False),
)

val_ds_full = PillDataset(
    split='val',
    transforms=val_transform(img_size),
    annotations=annotations,
    image_files=val_names,
    category_to_label=category_to_label,
)
loader = DataLoader(val_ds_full, batch_size=8, shuffle=False, collate_fn=collate_fn)

all_preds, all_targets = [], []
model.eval()
with torch.no_grad():
    for images, targets in loader:
        batch = torch.stack(images).to(device)
        raw = predict_batch(model, batch, device)
        preds = postprocess_raw_outputs(raw, image_ids=[t['image_id'] for t in targets], config=raw_post_cfg)
        all_preds.extend(preds)
        all_targets.extend({'boxes': t['boxes'], 'labels': t['labels']} for t in targets)
print('collected:', len(all_preds))

conf_range = np.arange(0.05, 0.91, 0.05)
f1_vals, prec_vals, rec_vals = [], [], []
for conf in conf_range:
    stats = compute_per_class_f1(all_preds, all_targets, num_classes=nc, conf_threshold=float(conf))
    if stats:
        f1_vals.append(np.mean([v['f1'] for v in stats.values()]))
        prec_vals.append(np.mean([v['precision'] for v in stats.values()]))
        rec_vals.append(np.mean([v['recall'] for v in stats.values()]))
    else:
        f1_vals.append(0.0); prec_vals.append(0.0); rec_vals.append(0.0)

best_idx = int(np.argmax(f1_vals))
best_conf = float(conf_range[best_idx])

plt.figure(figsize=(9, 5))
plt.plot(conf_range, f1_vals, label='F1', linewidth=2)
plt.plot(conf_range, prec_vals, label='Precision', linestyle='--')
plt.plot(conf_range, rec_vals, label='Recall', linestyle='--')
plt.axvline(best_conf, color='blue', linestyle=':', label=f'best conf={best_conf:.2f}')
plt.axvline(current_conf, color='red', linestyle='--', label=f'current conf={current_conf:.2f}')
plt.grid(True, alpha=0.3); plt.legend(); plt.xlabel('conf threshold')
plt.title(f'F1 threshold sweep - {EXP_NAME}')
plt.show()

print(f'current conf: {current_conf:.2f}')
print(f'best conf: {best_conf:.2f}, F1={f1_vals[best_idx]:.4f}')


## 6. Kaggle 제출

In [ ]:
from pathlib import Path
import pandas as pd

model_label = 'ema' if USE_EMA_FOR_PREDICT else 'raw'
chosen_csv = Path(f'outputs/submissions/submission_{EXP_NAME}_{model_label}.csv')
if not chosen_csv.exists():
    chosen_csv = Path('outputs/submissions/submission_v1.csv')

if chosen_csv.exists():
    df = pd.read_csv(chosen_csv)
    print('제출 파일:', chosen_csv)
    print('행 수:', len(df), '/ 이미지 수:', df['image_id'].nunique())
    display(df.head())
else:
    print('제출 CSV 없음 — 예측 셀을 먼저 실행하세요')


## 5-3. 산출물 Drive 백업


In [ ]:
import shutil
from pathlib import Path

SHARE_DIR.mkdir(parents=True, exist_ok=True)
files_to_copy = [
    Path(CONFIG_PATH),
    log_path if 'log_path' in globals() and log_path is not None else None,
    Path(f'outputs/logs/f1_{EXP_NAME}.csv'),
    BEST_CKPT if 'BEST_CKPT' in globals() else None,
    Path('outputs/predictions/predictions.json'),
    chosen_csv if 'chosen_csv' in globals() else Path('outputs/submissions/submission_v1.csv'),
]

for src in files_to_copy:
    if src is not None and Path(src).exists():
        dst = SHARE_DIR / Path(src).name
        shutil.copy2(src, dst)
        print('copied:', dst)
    else:
        print('missing/skip:', src)

zip_path = shutil.make_archive(str(SHARE_DIR), 'zip', SHARE_DIR)
print('zip:', zip_path)


In [ ]:
!kaggle competitions submit     -c {COMPETITION_ID}     -f {chosen_csv}     -m "{SUBMIT_MESSAGE} [{model_label}]"

print(f'제출 완료! ({model_label} 모델)')


In [ ]:
!kaggle competitions submissions -c {COMPETITION_ID}